In [1]:
import os
from pathlib import Path

root = Path().resolve().parent
os.chdir(root)
os.getcwd()

'C:\\Users\\Adesh\\OneDrive\\Documents\\ML NLP\\Projects\\03 End-to-End Evaluation and Benchmarking Framework for LLM Outputs and Retrieval-Based NLP Systems'

In [2]:
import json

with open("artifacts/eval_results.json", "r") as f:
    eval_results = json.load(f)

eval_results[0]

{'id': '572817584b864d1900164463',
 'question': 'Greater London is divided into what two groups of boroughs?',
 'ground_truth': ['Inner London and Outer London'],
 'baseline_answer': 'Greater London is divided into two groups of boroughs: the Inner London boroughs and the Outer London boroughs.',
 'rag_answer': 'Greater London is divided into Inner London and Outer London.',
 'retrieved_chunks': ['Outward urban expansion is now prevented by the Metropolitan Green Belt, although the built-up area extends beyond the boundary in places, resulting in a separately defined Greater London Urban Area. Beyond this is the vast London commuter belt. Greater London is split for some purposes into Inner London and Outer London. The city is split by the River Thames into North and South, with an informal central London area in its interior. The coordinates of the nominal centre of London, traditionally considered to be the original Eleanor Cross at Charing Cross near the junction of Trafalgar Square

In [3]:
import string

In [4]:
def norm_text(text):
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = " ".join(text.split())
    return text

In [5]:
sample_text = "Hello! World, it's sunny?"
norm_text(sample_text)

'hello world its sunny'

In [6]:
#exact match EM
def exact_match(prediction, ground_truths):
    norm_pred = norm_text(prediction)
    norm_ground_truths = [norm_text(gt) for gt in ground_truths]
    return int(norm_pred in norm_ground_truths)

In [7]:
sample = eval_results[0]
print("Baseline EM:")
print(exact_match(sample["baseline_answer"], sample["ground_truth"]))
print("RAG EM:")
print(exact_match(sample["rag_answer"], sample["ground_truth"]))

Baseline EM:
0
RAG EM:
0


In [8]:
#semantic similarity
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

In [10]:
def semantic_sim(prediction, ground_truths):
    pred_embedding = embedding_model.encode([prediction])
    similarities = []
    
    for gt in ground_truths:
        gt_embedding = embedding_model.encode([gt])
        similarity = cosine_similarity(pred_embedding, gt_embedding)[0][0]
        similarities.append(similarity)
    
    return max(similarities)

In [11]:
sample = eval_results[0]
print("Baseline Semantic Similarity:")
print(semantic_sim(sample["baseline_answer"], sample["ground_truth"]))
print("RAG Semantic Similarity:")
print(semantic_sim(sample["rag_answer"], sample["ground_truth"]))

Baseline Semantic Similarity:
0.61314636
RAG Semantic Similarity:
0.8436476


In [12]:
#retrieval precision at k
def is_relevant(chunk_text, ground_truths):
    chunk_text = norm_text(chunk_text)
    
    normalized_groud_truths = [norm_text(gt) for gt in ground_truths]

    for gt in normalized_groud_truths:
        if gt in chunk_text:
            return True
    return False

In [13]:
def precision_at_k(retrieved_chunks, ground_truths):
    relevance_count = 0

    for chunk in retrieved_chunks:
        if is_relevant(chunk, ground_truths):
            relevance_count += 1
    
    return relevance_count/len(retrieved_chunks)

In [14]:
sample = eval_results[0]
print("Baseline Semantic Similarity:")
print(semantic_sim(sample["baseline_answer"], sample["ground_truth"]))
print("RAG Semantic Similarity:")
print(semantic_sim(sample["rag_answer"], sample["ground_truth"]))
print("Precision@k:")
print(precision_at_k(sample["retrieved_chunks"], sample["ground_truth"]))

Baseline Semantic Similarity:
0.61314636
RAG Semantic Similarity:
0.8436476
Precision@k:
0.3333333333333333


In [15]:
#recall at k
def recall_at_k(retrieved_chunks, ground_truths):
    for chunk in retrieved_chunks:
        if is_relevant(chunk, ground_truths):
            return 1
    return 0

In [16]:
sample = eval_results[0]
print("Recall@k:")
print(recall_at_k(sample["retrieved_chunks"], sample["ground_truth"]))

Recall@k:
1


In [17]:
#mean reciprocal rank
def reciprocal_rank(retrieved_chunks, ground_truths):
    for i, chunk in enumerate (retrieved_chunks, 1):
        if is_relevant(chunk, ground_truths):
            return 1/i
    return 0

In [18]:
sample = eval_results[0]
print("MRR:")
print(reciprocal_rank(sample["retrieved_chunks"], sample["ground_truth"]))

MRR:
1.0


In [19]:
#benchmark aggregating over all samples from subset
baseline_em_scores = []
rag_em_scores = []

baseline_semantic_sim_scores = []
rag_semantic_sim_scores = []

precision_at_k_scores = []
recall_at_k_scores = []
mrr_scores = []

In [20]:
for sample in eval_results:
    baseline_em_scores.append(exact_match(sample["baseline_answer"], sample["ground_truth"]))
    rag_em_scores.append(exact_match(sample["rag_answer"], sample["ground_truth"]))

    baseline_semantic_sim_scores.append(semantic_sim(sample["baseline_answer"], sample["ground_truth"]))
    rag_semantic_sim_scores.append(semantic_sim(sample["rag_answer"], sample["ground_truth"]))

    precision_at_k_scores.append(precision_at_k(sample["retrieved_chunks"], sample["ground_truth"]))
    recall_at_k_scores.append(recall_at_k(sample["retrieved_chunks"], sample["ground_truth"]))
    mrr_scores.append(reciprocal_rank(sample["retrieved_chunks"], sample["ground_truth"]))

In [21]:
print(f"Baseline EM Score: ", sum(baseline_em_scores)/len(baseline_em_scores))
print(f"RAG EM Score: ", sum(rag_em_scores)/len(rag_em_scores))
print(f"\nBaseline Semantic Sim Score: ", sum(baseline_semantic_sim_scores)/len(baseline_semantic_sim_scores))
print(f"RAG EM Score: ", sum(rag_semantic_sim_scores)/len(rag_semantic_sim_scores))
print(f"\nPrecision@k: ", sum(precision_at_k_scores)/len(precision_at_k_scores))
print(f"Recall@k: ", sum(recall_at_k_scores)/len(recall_at_k_scores))
print(f"MRR: ", sum(mrr_scores)/len(mrr_scores))

Baseline EM Score:  0.0
RAG EM Score:  0.0

Baseline Semantic Sim Score:  0.3480887860059738
RAG EM Score:  0.5220109134912491

Precision@k:  0.3
Recall@k:  0.9
MRR:  0.9


In [22]:
#LLM as judge
def get_judge_prompt(question, ground_truth, answer):
    return f"""
        You are evaluating a QA system answer.
        Question: {question}
        Ground Truth Answer: {ground_truth}
        Model Answer: {answer}
        Now, evaluate the Model Answer on the following: Correctness, Completeness and FLuency
        Give a score ranging from 1 to 5.
        Respond ONLY in this format:
        Correctness: <score>
        Completeness: <score>
        Fluency: <score>
        """

In [23]:
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
client = OpenAI(api_key = os.getenv("OPENAI_API_KEY"))

In [26]:
def llm_judge(question, ground_truth, answer):
    prompt = get_judge_prompt(question, ground_truth, answer)
    response = client.chat.completions.create(
        model = "gpt-4o-mini",
        messages = [{"role": "user", "content": prompt}],
        temperature = 0
    )

    return response.choices[0].message.content

In [27]:
#checking llm
sample = eval_results[0]
baseline_judge = llm_judge(sample["question"], sample["ground_truth"], sample["baseline_answer"])
print("Baseline Judge Evaluation:")
print(baseline_judge)

Baseline Judge Evaluation:
Correctness: 5  
Completeness: 5  
Fluency: 5  


In [28]:
rag_judge = llm_judge(sample["question"], sample["ground_truth"], sample["rag_answer"])
print("RAG Judge Evaluation:")
print(rag_judge)

RAG Judge Evaluation:
Correctness: 5  
Completeness: 5  
Fluency: 5  


In [29]:
#structing the LLM output
import re
def parse_judge_scores(judge_response):
    correctness = re.search(r"Correctness:\s*(\d)", judge_response)
    completeness = re.search(r"Completeness:\s*(\d)", judge_response)
    fluency = re.search(r"Fluency:\s*(\d)", judge_response)

    return {
        "correctness": int(correctness.group(1)),
        "completeness": int(completeness.group(1)),
        "fluency": int(fluency.group(1)) 
    }

In [30]:
parsed = parse_judge_scores(baseline_judge)
parsed

{'correctness': 5, 'completeness': 5, 'fluency': 5}

In [31]:
baseline_correctness = []
baseline_completeness = []
baseline_fluency = []

rag_correctness= []
rag_completness = []
rag_fluency = []

In [32]:
for sample in eval_results:
    baseline_judge = llm_judge(sample["question"], sample["ground_truth"], sample["baseline_answer"])
    baseline_scores = parse_judge_scores(baseline_judge)
    baseline_correctness.append(baseline_scores["correctness"])
    baseline_completeness.append(baseline_scores["completeness"])
    baseline_fluency.append(baseline_scores["fluency"])

    rag_judge = llm_judge(sample["question"], sample["ground_truth"], sample["rag_answer"])
    rag_scores = parse_judge_scores(rag_judge)
    rag_correctness.append(rag_scores["correctness"])
    rag_completness.append(rag_scores["completeness"])
    rag_fluency.append(rag_scores["fluency"])

In [33]:
print("Judge Scores:")
print("Baseline Correctness: ", sum(baseline_correctness)/len(baseline_correctness))
print("Baseline Completeness: ", sum(baseline_completeness)/len(baseline_completeness))
print("Baseline Fluency: ", sum(baseline_fluency)/len(baseline_fluency))
print("\nRAG Correctness: ", sum(rag_correctness)/len(rag_correctness))
print("RAG Completeness: ", sum(rag_completness)/len(rag_completness))
print("Baseline Fluency: ", sum(rag_fluency)/len(rag_fluency))

Judge Scores:
Baseline Correctness:  3.9
Baseline Completeness:  3.8
Baseline Fluency:  5.0

RAG Correctness:  4.5
RAG Completeness:  4.4
Baseline Fluency:  5.0
